# Model multi-objective optimization problems with the `MultiObjectiveProblem` class

## Introduction

This guide demonstrates how to model problems with **several objective functions at once** by using the `MultiObjectiveProblem` class of the **Optimization Mapper Qiskit addon**.

Many real problems ask for more than one thing at the same time, and the things they ask for conflict: you want a portfolio with high return *and* low risk, or a schedule that is short *and* cheap. No single assignment is usually best for every objective at once, so the goal is not one optimum but the **trade-off surface** between the objectives. `MultiObjectiveProblem` lets you write such a model down and translate each of its objectives.

Where `OptimizationProblem` has one objective, `MultiObjectiveProblem` holds an ordered collection of them. You can express the problem in the following general form:

$$
\begin{align}
\text{optimize}\quad& \bigl(f_1(x),\; f_2(x),\; \dots,\; f_p(x)\bigr) \\
\text{subject to}\quad&
\sum_{d=1}^{D} \;\; \sum_{S \subseteq \{1,\dots,n\}, |S|=d} a^{(j)}_{S} \prod_{i \in S} x_i \;\; \leq b_j, \quad j=1,\dots,m \\
& l_i \leq x_i \leq u_i, \quad i=1,\dots,n,
\end{align}
$$

where each $f_k$ is a polynomial of any degree up to $D$, exactly like the single objective of an `OptimizationProblem`, and **each $f_k$ has its own sense**: you can minimize some objectives and maximize others within the same problem.

`MultiObjectiveProblem` is a **modeling** class. It stores your objectives and translates each of them into an Ising Hamiltonian, but it deliberately does not combine them for you: it does not scalarize them into a weighted sum, and it does not compute a Pareto front. Choosing how to explore the trade-off is left to you and to the algorithm you run.

## Relationship to `OptimizationProblem`

`MultiObjectiveProblem` and `OptimizationProblem` are siblings: both derive from `BaseOptimizationProblem`, which holds the variables, the constraints, the problem name, and the status. Only the handling of the objective differs between them.

Everything in [Model quantum optimization problems with the `OptimizationProblem` class](01_optimization_problem.ipynb) that does not concern the objective therefore works on a `MultiObjectiveProblem` unchanged, with the same method names and the same behavior:

- Creating **binary**, **integer**, **continuous**, and **spin** variables.
- Adding, reading, and removing **linear**, **quadratic**, and **higher-order constraints**.
- `prettyprint()` for a readable rendering of the whole model.
- `get_feasibility_info()` and `is_feasible()` to check a candidate solution.
- `substitute_variables()` to replace variables with constants or with other variables.
- `clear()` to empty the model.

This guide does not repeat any of that. It concentrates on what is different: how you **add and inspect several objectives**, and how you **translate them into Ising Hamiltonians**.

## Construct a `MultiObjectiveProblem`

Start from an empty model, as you would with `OptimizationProblem`. Note that `prettyprint()` reports `No objectives`. An `OptimizationProblem` always has exactly one objective, but a `MultiObjectiveProblem` starts with none, and a problem with zero objectives is legal.

In [1]:
from qiskit_addon_opt_mapper import MultiObjectiveProblem

# make an empty problem
problem = MultiObjectiveProblem("item selection")
print(problem.prettyprint())

Problem name: item selection

  No objectives

Subject to
  No constraints

  No variables



The running example is a small selection problem: three items, each either taken or left, which gives three binary variables. You declare variables exactly as in the single-objective case.

In [2]:
# Add variables
problem.binary_var(name="x0")
problem.binary_var(name="x1")
problem.binary_var(name="x2")
print(problem.prettyprint())

Problem name: item selection

  No objectives

Subject to
  No constraints

  Binary variables (3)
    x0 x1 x2



Constraints are added exactly as they are on an `OptimizationProblem`. The next cell is the only reminder of that in this guide: you may take at most two of the three items.

In [3]:
# Add a linear constraint, exactly as for a single-objective problem
problem.linear_constraint(linear={"x0": 1, "x1": 1, "x2": 1}, sense="<=", rhs=2, name="at most two")
print(problem.prettyprint())

Problem name: item selection

  No objectives

Subject to
  Linear constraints (1)
    x0 + x1 + x2 <= 2  'at most two'

  Binary variables (3)
    x0 x1 x2



## Add objectives

This is the central difference. Instead of `minimize()` or `maximize()`, which *replace* the single objective of an `OptimizationProblem`, you call `add_objective()`, which **appends** one more objective to the collection.

`add_objective()` takes the same description of a polynomial as `minimize()` does (`constant`, `linear`, `quadratic`, and `higher_order`) plus two arguments of its own:

- `sense`, because each objective has its own. You can pass an `ObjSense` member or a string: `"min"`, `"minimize"`, `"max"`, and `"maximize"` are accepted.
- `name`, to identify the objective later. If you omit it, the objective is named automatically `f0`, `f1`, and so on, in the same way that constraints are named `c0`, `c1`, and so on.

The following example minimizes the total cost of the selected items and maximizes their total value. The two pull in opposite directions, which is the situation `MultiObjectiveProblem` is designed for. The first objective passes its sense as an `ObjSense` member and the second passes it as the string `"max"`.

In [4]:
from qiskit_addon_opt_mapper.problems import ObjSense

# Add two objectives with opposite senses
cost = problem.add_objective(linear={"x0": 5, "x1": 3, "x2": 8}, sense=ObjSense.MINIMIZE)
value = problem.add_objective(linear={"x0": 4, "x1": 2, "x2": 7}, sense="max")
print(problem.prettyprint())

Problem name: item selection

Minimize
  5*x0 + 3*x1 + 8*x2 'f0'
Maximize
  4*x0 + 2*x1 + 7*x2 'f1'

Subject to
  Linear constraints (1)
    x0 + x1 + x2 <= 2  'at most two'

  Binary variables (3)
    x0 x1 x2



`prettyprint()` shows one `Minimize` or `Maximize` block per objective, each followed by its name in quotes. Neither objective above was given a name, so they became `f0` and `f1`.

`add_objective()` returns the objective it just added, and that object is the one held by the problem, not a copy. You can read its name, sense, and coefficients from it.

In [5]:
print("cost  ->", cost.name, "|", cost.sense.name.lower())
print("value ->", value.name, "|", value.sense.name.lower())
print("same object as in the problem:", cost is problem.objectives[0])

cost  -> f0 | minimize
value -> f1 | maximize
same object as in the problem: True


Objectives are full polynomials, so an objective can carry quadratic and higher-order terms, just like the single objective of an `OptimizationProblem`. The higher-order terms use the same nested dictionary as in the single-objective guide: the outer key is the degree and the inner dictionary maps a tuple of variable names to its coefficient. The third objective below has an explicit name and a cubic term.

In [6]:
# Add a named objective with quadratic and higher-order terms
problem.add_objective(
    quadratic={("x0", "x1"): 1.5},
    higher_order={3: {("x0", "x1", "x2"): 2.0}},
    name="risk",
)
print(problem.prettyprint())

Problem name: item selection

Minimize
  5*x0 + 3*x1 + 8*x2 'f0'
Maximize
  4*x0 + 2*x1 + 7*x2 'f1'
Minimize
  2*x0*x1*x2 + 1.5*x0*x1 'risk'

Subject to
  Linear constraints (1)
    x0 + x1 + x2 <= 2  'at most two'

  Binary variables (3)
    x0 x1 x2



## Inspect and remove objectives

You read the collection of objectives through the same kind of interface that the constraint families have:

- `get_num_objectives()` counts the objectives.
- `objectives` is the list of objectives in the order you added them.
- `objectives_index` maps each name to its index.
- `get_objective()` accepts either an index or a name.

In [7]:
print("number of objectives:", problem.get_num_objectives())
print("name -> index:       ", problem.objectives_index)
print("by index 1:          ", problem.get_objective(1))
print("by name 'risk':      ", problem.get_objective("risk"))

number of objectives: 3
name -> index:        {'f0': 0, 'f1': 1, 'risk': 2}
by index 1:           maximize 4*x0 + 2*x1 + 7*x2 'f1'
by name 'risk':       minimize 2*x0*x1*x2 + 1.5*x0*x1 'risk'


`remove_objective()` also accepts an index or a name. Removing an objective shifts the indices of the objectives after it, but it does **not** change their names. Removing `f0` below moves `f1` to index 0 and `risk` to index 1, while both keep their names.

In [8]:
problem.remove_objective("f0")  # equivalently: problem.remove_objective(0)
print("number of objectives:", problem.get_num_objectives())
print("name -> index:       ", problem.objectives_index)

number of objectives: 2
name -> index:        {'f1': 0, 'risk': 1}


An objective's `name` is read-only, so there is no rename operation: assigning to it raises an `AttributeError`.

In [9]:
try:
    problem.get_objective("risk").name = "variance"
except AttributeError as e:
    print(f"Assigning to name raised {type(e).__name__}")

Assigning to name raised AttributeError


To rename an objective, remove it and add it again under the new name. `add_objective()` always appends, so the re-added objective goes to the end of the list. If the objective was not already last, the order of `objectives` changes, and so does the order of the Hamiltonians that `to_ising()` returns. Here `risk` is already last, so the order is unchanged.

In [10]:
problem.remove_objective("risk")
problem.add_objective(
    quadratic={("x0", "x1"): 1.5},
    higher_order={3: {("x0", "x1", "x2"): 2.0}},
    name="variance",
)
print("name -> index:", problem.objectives_index)

name -> index: {'f1': 0, 'variance': 1}


## Get the Ising Hamiltonians

To run an objective on a quantum computer you need its Ising Hamiltonian. `MultiObjectiveProblem.to_ising()` produces **one Hamiltonian per objective**, returned as a list of `(operator, offset)` pairs in objective order. In contrast, `OptimizationProblem.to_ising()` returns a single such pair.

`to_ising()` requires a problem with **no constraints** whose variables are **all binary or all spin**, so the model above does not qualify, because it has a constraint. This section builds a small unconstrained, all-binary problem from scratch, using only the `MultiObjectiveProblem` class imported earlier. Both objectives get explicit names to make the output easy to follow.

In [11]:
# An unconstrained, all-binary version of the same trade-off
ising_problem = MultiObjectiveProblem("item selection, unconstrained")
ising_problem.binary_var(name="x0")
ising_problem.binary_var(name="x1")
ising_problem.binary_var(name="x2")

ising_problem.add_objective(linear={"x0": 5, "x1": 3, "x2": 8}, sense="min", name="cost")
ising_problem.add_objective(linear={"x0": 4, "x1": 2, "x2": 7}, sense="max", name="value")
print(ising_problem.prettyprint())

Problem name: item selection, unconstrained

Minimize
  5*x0 + 3*x1 + 8*x2 'cost'
Maximize
  4*x0 + 2*x1 + 7*x2 'value'

Subject to
  No constraints

  Binary variables (3)
    x0 x1 x2



The returned list has one entry for every objective, in the same order as `objectives`, so you can zip the two together to see which Hamiltonian belongs to which objective. In each operator, the rightmost character of a Pauli string acts on `x0`, the next one on `x1`, and so on.

In [12]:
hamiltonians = ising_problem.to_ising()
print(f"{len(hamiltonians)} Hamiltonians for {ising_problem.get_num_objectives()} objectives\n")

for objective, (operator, offset) in zip(ising_problem.objectives, hamiltonians, strict=True):
    print(f"objective '{objective.name}' ({objective.sense.name.lower()})")
    print(f"  operator: {operator}")
    print(f"  offset:   {offset}")
    print()

2 Hamiltonians for 2 objectives

objective 'cost' (minimize)
  operator: SparsePauliOp(['IIZ', 'IZI', 'ZII'],
              coeffs=[-2.5+0.j, -1.5+0.j, -4. +0.j])
  offset:   8.0

objective 'value' (maximize)
  operator: SparsePauliOp(['IIZ', 'IZI', 'ZII'],
              coeffs=[2. +0.j, 1. +0.j, 3.5+0.j])
  offset:   -6.5



Each Hamiltonian is a **minimization**. The sense of a maximization objective is folded into its own Hamiltonian by negating it: `value` is maximized, so its Hamiltonian describes minimizing $-(4x_0 + 2x_1 + 7x_2)$. That is why its offset is negative, and why its operator has the opposite signs to that of `cost`.

If a problem has constraints, or has variables that are not all binary or all spin, `to_ising()` raises an `OptimizationError`. The following cell builds a constrained problem and shows the error.

In [13]:
from qiskit_addon_opt_mapper import OptimizationError

constrained = MultiObjectiveProblem("constrained")
constrained.binary_var(name="x0")
constrained.binary_var(name="x1")
constrained.linear_constraint(linear={"x0": 1, "x1": 1}, sense="<=", rhs=1, name="at most one")
constrained.add_objective(linear={"x0": 1, "x1": 2}, name="cost")

try:
    constrained.to_ising()
except OptimizationError as e:
    print(f"Error: {e}")

Error: There must be no constraint in the problem. You can use `OptimizationProblemToQubo` converter to convert constraints to penalty terms of the objective function.


The error message suggests running `OptimizationProblemToQubo` first, which is the right advice for a single-objective `OptimizationProblem`. Be aware of a limitation in this release: the converters in [Use converters for optimization problems](02_converters_for_optimization_problems.ipynb) work only on single-objective `OptimizationProblem` instances, so you cannot use them on a `MultiObjectiveProblem`. Most of them fail with an `AttributeError` about a missing `objective` attribute if you try. For now, build a `MultiObjectiveProblem` whose objectives are already unconstrained and binary (or spin), as in the earlier example.